# Notebook 01 — Data Collection & Cleaning

**Assignment 1B — Enterprise Domain LLM**  
**Author:** Abhinav Mandloi

This notebook documents the full data pipeline:
1. PDF collection (Ubuntu, Linux troubleshooting, AWS EC2 manuals)
2. Text extraction with PyMuPDF
3. Multi-stage text cleaning (length filter → whitespace → deduplication → language filter)
4. Corpus statistics and visualisation

---

In [ ]:
import os, sys, json, hashlib, re, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Project root
PROJECT_ROOT = Path('/Users/abhinavmandloi/Documents/Sem3Projects/LLM Assingment 1/enterprise-domain-llm')
sys.path.insert(0, str(PROJECT_ROOT))

RAW_PDF_DIR   = PROJECT_ROOT / 'data' / 'raw_pdfs'
EXTRACTED_DIR = PROJECT_ROOT / 'data' / 'extracted_text'
CORPUS_DIR    = PROJECT_ROOT / 'data' / 'domain_corpus'

for d in [RAW_PDF_DIR, EXTRACTED_DIR, CORPUS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready.')
print(f'  raw_pdfs       : {RAW_PDF_DIR}')
print(f'  extracted_text : {EXTRACTED_DIR}')
print(f'  domain_corpus  : {CORPUS_DIR}')

## 1.1 PDF Source Documentation

The domain corpus is built from publicly available IT administration and troubleshooting documents covering:

| # | Document | Domain | Source |
|---|----------|--------|---------|
| 1 | Ubuntu Server Guide 22.04 | Linux/Ubuntu | ubuntu.com/server/docs |
| 2 | AWS EC2 User Guide | Cloud/AWS | docs.aws.amazon.com |
| 3 | Linux Filesystem Hierarchy Standard | Linux | pathname.com/fhs |
| 4 | Red Hat System Administration Guide | Linux | access.redhat.com |
| 5 | AWS CLI Reference | Cloud/AWS | docs.aws.amazon.com/cli |

Documents are stored in `data/raw_pdfs/`. The extraction below is idempotent: it skips PDFs already extracted.

In [ ]:
# --- PDF Text Extraction with PyMuPDF ---
try:
    import fitz  # PyMuPDF
    PYMUPDF_OK = True
    print(f'PyMuPDF version: {fitz.__version__}')
except ImportError:
    PYMUPDF_OK = False
    print('WARNING: PyMuPDF not installed — extraction will be skipped.')

def extract_pdf(pdf_path: Path, out_dir: Path) -> dict:
    """Extract text from a PDF file page by page. Returns stats dict."""
    out_path = out_dir / (pdf_path.stem + '.txt')
    if out_path.exists():
        text = out_path.read_text(encoding='utf-8')
        return {'file': pdf_path.name, 'pages': text.count('\f') + 1, 'chars': len(text), 'skipped': True}
    doc = fitz.open(str(pdf_path))
    pages_text = []
    for page in doc:
        pages_text.append(page.get_text('text'))
    full_text = '\f'.join(pages_text)  # form-feed as page separator
    out_path.write_text(full_text, encoding='utf-8')
    return {'file': pdf_path.name, 'pages': len(pages_text), 'chars': len(full_text), 'skipped': False}

pdf_files = sorted(RAW_PDF_DIR.glob('*.pdf'))
extraction_stats = []

if PYMUPDF_OK and pdf_files:
    for pdf in pdf_files:
        stat = extract_pdf(pdf, EXTRACTED_DIR)
        extraction_stats.append(stat)
        flag = '(cached)' if stat['skipped'] else ''
        print(f"  {stat['file']}: {stat['pages']} pages, {stat['chars']:,} chars {flag}")
    df_extract = pd.DataFrame(extraction_stats)
    print('\nExtraction summary:')
    print(df_extract.to_string(index=False))
elif not pdf_files:
    print('No PDF files found in raw_pdfs/. Creating synthetic corpus for demonstration.')
    # Create synthetic domain text so pipeline demonstrates correctly
    synthetic_texts = [
        ('ubuntu_ntfs_guide.txt', """
Ubuntu Server NTFS Troubleshooting Guide
=========================================
NTFS drives on Ubuntu require the ntfs-3g package. To mount an NTFS partition:
  sudo apt-get install ntfs-3g
  sudo mount -t ntfs-3g /dev/sdb1 /mnt/data
If mount fails with 'failed to read last sector', run: ntfsfix /dev/sdb1
Common error codes: NTFS signature missing (corrupted), Metadata kept in Windows cache (disable Fast Startup in Windows).
To check filesystem: sudo ntfsfix -d /dev/sdb1
Persistent mount: add to /etc/fstab: /dev/sdb1 /mnt/data ntfs-3g defaults,uid=1000,gid=1000 0 0
Verify UUID for stability: blkid /dev/sdb1
"""),
        ('linux_io_troubleshooting.txt', """
Linux Production Server I/O and CPU Troubleshooting
===================================================
High disk I/O investigation:
  iostat -xz 1          # extended stats every 1 second
  iotop -ao             # processes with accumulated I/O
  dstat -cdlmnpsy 1     # combined stats
  pidstat -d 2          # per-process disk stats
CPU usage analysis:
  top                   # real-time process viewer
  htop                  # interactive process viewer
  mpstat -P ALL 1       # per-core utilisation
  perf top              # kernel-level profiling
  strace -p PID         # syscall tracing
Identify I/O-bound processes: look for high wa% in top (iowait).
Tune I/O scheduler: cat /sys/block/sda/queue/scheduler; echo 'mq-deadline' > /sys/block/sda/queue/scheduler
Disk bottleneck: fio, dd if=/dev/zero of=/tmp/test bs=1G count=1 oflag=dsync
"""),
        ('aws_ec2_connectivity.txt', """
AWS EC2 Connectivity and SSH Troubleshooting
============================================
SSH access issues:
  - Verify Security Group: inbound rule TCP 22 from your IP.
  - Check key pair: ssh -i ~/.ssh/key.pem ec2-user@public-ip -v
  - Verify instance state: aws ec2 describe-instances --instance-ids i-xxx
  - Check System Log: AWS Console > EC2 > Actions > Monitor > Get System Log
Connectivity checklist:
  1. Instance running: EC2 Console shows green 'running'
  2. Public IP or Elastic IP attached
  3. Internet Gateway attached to VPC
  4. Route table: 0.0.0.0/0 -> igw-xxx
  5. Security Group: inbound SSH (22) open
  6. NACL: default allows all, custom NACLs may block
  7. Host firewall (iptables/ufw) not blocking port 22
If instance unreachable: use EC2 Instance Connect (browser SSH) or Session Manager (SSM).
EC2 serial console: aws ec2 enable-serial-console-access; connect via console.
"""),
    ]
    for fname, text in synthetic_texts:
        (EXTRACTED_DIR / fname).write_text(text.strip(), encoding='utf-8')
    print(f'Created {len(synthetic_texts)} synthetic domain text files in extracted_text/')
else:
    print('PyMuPDF not available — skipping real PDF extraction.')

## 1.2 Text Cleaning Pipeline

Four sequential cleaning stages:

| Stage | Operation | Purpose |
|-------|-----------|----------|
| 1 | Minimum length filter (≥ 100 chars) | Remove page headers/footers, empty pages |
| 2 | Whitespace normalisation | Collapse multiple spaces/newlines, strip non-printable chars |
| 3 | Exact deduplication (SHA-256) | Remove duplicate passages |
| 4 | Language filter (English only) | Discard non-English content |

In [ ]:
import unicodedata

try:
    from langdetect import detect, LangDetectException
    LANGDETECT_OK = True
except ImportError:
    LANGDETECT_OK = False
    print('langdetect not installed — language filter will be skipped.')

MIN_CHARS = 100

# Collect all extracted text segments (split by paragraph / form-feed)
segments = []
for txt_file in sorted(EXTRACTED_DIR.glob('*.txt')):
    raw = txt_file.read_text(encoding='utf-8', errors='replace')
    # Split on page boundaries or double newlines
    parts = re.split(r'\f|\n{3,}', raw)
    segments.extend([(txt_file.name, p.strip()) for p in parts if p.strip()])

print(f'Total segments before cleaning : {len(segments):,}')
stage_counts = {'raw': len(segments)}

# Stage 1: Length filter
segments = [(f, s) for f, s in segments if len(s) >= MIN_CHARS]
stage_counts['length_filter'] = len(segments)
print(f'After length filter (≥{MIN_CHARS} chars) : {len(segments):,}')

# Stage 2: Whitespace normalisation
def normalize_whitespace(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[^\S\n]+', ' ', text)      # collapse horiz whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)     # collapse excess newlines
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)  # strip control chars
    return text.strip()

segments = [(f, normalize_whitespace(s)) for f, s in segments]
segments = [(f, s) for f, s in segments if len(s) >= MIN_CHARS]
stage_counts['whitespace_norm'] = len(segments)
print(f'After whitespace normalisation : {len(segments):,}')

# Stage 3: Exact deduplication (SHA-256)
seen_hashes = set()
deduped = []
for f, s in segments:
    h = hashlib.sha256(s.encode()).hexdigest()
    if h not in seen_hashes:
        seen_hashes.add(h)
        deduped.append((f, s))
segments = deduped
stage_counts['deduplication'] = len(segments)
print(f'After exact deduplication     : {len(segments):,}')

# Stage 4: Language filter
if LANGDETECT_OK:
    def is_english(text: str) -> bool:
        try:
            return detect(text[:500]) == 'en'
        except LangDetectException:
            return False
    segments = [(f, s) for f, s in segments if is_english(s)]
else:
    print('  (language filter skipped)')

stage_counts['language_filter'] = len(segments)
print(f'After language filter (en)     : {len(segments):,}')
print(f'\nFinal corpus size: {len(segments):,} segments')

## 1.3 Corpus Statistics

In [ ]:
total_chars  = sum(len(s) for _, s in segments)
total_words  = sum(len(s.split()) for _, s in segments)
avg_seg_len  = total_chars / max(len(segments), 1)

seg_lengths  = [len(s) for _, s in segments]

stats = {
    'Total segments'      : f'{len(segments):,}',
    'Total characters'    : f'{total_chars:,}',
    'Total words (approx)': f'{total_words:,}',
    'Avg segment length'  : f'{avg_seg_len:.0f} chars',
    'Min segment length'  : f'{min(seg_lengths) if seg_lengths else 0}',
    'Max segment length'  : f'{max(seg_lengths) if seg_lengths else 0}',
}

df_stats = pd.DataFrame(list(stats.items()), columns=['Metric', 'Value'])
print('Corpus Statistics:')
print(df_stats.to_string(index=False))

# Save corpus
corpus_path = CORPUS_DIR / 'cleaned_corpus.jsonl'
with open(corpus_path, 'w', encoding='utf-8') as f:
    for source, text in segments:
        f.write(json.dumps({'source': source, 'text': text}) + '\n')
print(f'\nSaved {len(segments):,} segments to {corpus_path}')

## 1.4 Visualisation: Cleaning Pipeline & Segment Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: segment counts at each stage
ax1 = axes[0]
labels = list(stage_counts.keys())
values = list(stage_counts.values())
colours = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
bars = ax1.bar(labels, values, color=colours[:len(labels)])
ax1.set_title('Segments Remaining After Each Cleaning Stage', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Segments')
ax1.set_xlabel('Cleaning Stage')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:,}', ha='center', va='bottom', fontsize=9)
ax1.tick_params(axis='x', rotation=25)

# Histogram: segment length distribution
ax2 = axes[1]
if seg_lengths:
    ax2.hist(seg_lengths, bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
else:
    ax2.text(0.5, 0.5, 'No segments', ha='center', transform=ax2.transAxes)
ax2.set_title('Segment Length Distribution (chars)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Segment Length (characters)')
ax2.set_ylabel('Frequency')
ax2.axvline(avg_seg_len, color='red', linestyle='--', label=f'Mean ({avg_seg_len:.0f})')
ax2.legend()

plt.tight_layout()
charts_dir = PROJECT_ROOT / 'results' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(charts_dir / 'corpus_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/charts/corpus_stats.png')

## 1.5 Observations on the Cleaning Pipeline

**Length filter** is the most aggressive stage, removing page headers, footers, and partial table rows. These short fragments (<100 chars) carry little semantic content.

**Whitespace normalisation** standardises encoding artefacts that occur during PDF-to-text conversion (ligatures, soft hyphens, non-breaking spaces). No segments are removed, only cleaned.

**Exact deduplication** using SHA-256 efficiently removes repeated boilerplate (e.g., copyright notices, repeated headers).

**Language filter** ensures the corpus is strictly English, which is critical for model fine-tuning on English IT documentation.

The final corpus is saved as a JSONL file with `source` and `text` fields, ready for the instruction dataset generation step in Notebook 03.

---
**Next:** Notebook 02 — Baseline Model Evaluation